# Argoverse 2 GRU data scaling (Google Colab)

Runs the same agent-centric GRU configuration on 500, 5,000, and 20,000 training scenarios. All runs evaluate against the same 500 validation scenarios. Set a Colab GPU runtime before running.

The training and evaluation implementation remains in the repository scripts. This notebook only clones the project, mounts Drive, and invokes those scripts.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/<your-user>/motion-forecasting.git"  # Set this to the pushed repository URL.
REPO_DIR = Path("/content/motion-forecasting")
if not REPO_DIR.exists():
    if "<your-user>" in REPO_URL:
        raise ValueError("Set REPO_URL to the Git URL for this project, then rerun this cell.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

%cd /content/motion-forecasting
!pip install -e .

## Mount Google Drive and set dataset paths

Put the AV2 Motion Forecasting split directories in Drive so they contain at least 20,000 train Parquet files and exactly the same 500 validation Parquet files used by earlier runs. The notebook checks these counts before training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/av2/motion-forecasting")  # Change to your Drive folder.
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
RUN_DIR = DATA_ROOT / "experiments" / "gru_data_scaling"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def scenario_count(path):
    return len(list(path.rglob("scenario_*.parquet")))

train_count = scenario_count(TRAIN_DIR)
val_files = sorted(VAL_DIR.rglob("scenario_*.parquet"))
val_ids = {path.parent.name for path in val_files}
expected_val_ids = set((REPO_DIR / "data/val_scenario_ids.txt").read_text().splitlines())
val_count = len(val_files)
print(f"Train scenarios available: {train_count:,}")
print(f"Validation scenarios available: {val_count:,}")
assert train_count >= 20_000, "Drive train split must contain at least 20,000 scenarios."
assert val_count == 500 and val_ids == expected_val_ids, "Use exactly the fixed validation IDs in data/val_scenario_ids.txt."

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Runtime > Change runtime type.")
print(torch.cuda.get_device_name(0))

## Train, evaluate, and save each run

The seed, hidden dimension, loss, representation, validation set, batch size, and epoch count are held constant. Only the number of training scenarios changes. Per-epoch CSV files and learning-curve PNGs are saved beside the best validation-ADE checkpoint.

In [ ]:
import json

TRAIN_SCENARIOS = (500, 5_000, 20_000)
SEED = 42
EPOCHS = 20
BATCH_SIZE = 256
HIDDEN_DIM = 128
NUM_WORKERS = 4
results = []

for count in TRAIN_SCENARIOS:
    run_name = f"gru_{count}"
    checkpoint = RUN_DIR / f"{run_name}.pt"
    train_cmd = [
        sys.executable, "scripts/train_gru.py",
        "--train-data", str(TRAIN_DIR), "--val-data", str(VAL_DIR),
        "--device", "cuda", "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--hidden-dim", str(HIDDEN_DIM),
        "--seed", str(SEED), "--num-workers", str(NUM_WORKERS),
        "--train-scenarios", str(count), "--output", str(checkpoint),
    ]
    subprocess.run(train_cmd, cwd=REPO_DIR, check=True)

    metrics_path = RUN_DIR / f"{run_name}.metrics.json"
    eval_cmd = [
        sys.executable, "scripts/evaluate_gru.py",
        "--data", str(VAL_DIR), "--checkpoint", str(checkpoint),
        "--device", "cuda", "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS), "--metrics-output", str(metrics_path),
    ]
    subprocess.run(eval_cmd, cwd=REPO_DIR, check=True)
    results.append(json.loads(metrics_path.read_text()))

summary = [{"train_scenarios": count, **result} for count, result in zip(TRAIN_SCENARIOS, results)]
summary_path = RUN_DIR / "data_scaling_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary_frame = pd.DataFrame([
    {"Model": "GRU", "Train scenes": row["train_scenarios"],
     "ADE (m)": row["ade_m"], "FDE (m)": row["fde_m"]}
    for row in summary
])
display(summary_frame)

fig, ax = plt.subplots(figsize=(8, 5))
for count in TRAIN_SCENARIOS:
    curve = pd.read_csv(RUN_DIR / f"gru_{count}.csv")
    ax.plot(curve["epoch"], curve["val_ade"], marker="o", label=f"{count:,} train scenes")
ax.set(title="GRU validation ADE by training-set size", xlabel="Epoch", ylabel="ADE (m)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
curve_path = RUN_DIR / "data_scaling_val_ade.png"
fig.savefig(curve_path, dpi=160)
plt.show()
print(f"Saved summary to {summary_path}")
print(f"Saved ADE plot to {curve_path}")